[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/modes/modes.ipynb)

# Vibration Modes of a Suspended Plate

This example finds the natural vibrations of a rigid plate hanging from springs in space. Pushed and let go, the plate swings in a few fixed patterns, each at its own frequency: sliding sideways, bouncing up and down, rocking. Which patterns, and at which frequencies, depends on where the springs hold the plate and how its mass is spread; one more spring set at an angle mixes the patterns together.

A motion of the plate is a twist and a load on it a forque. A spring is the line from its anchor to its attachment, and paired with a twist, `Twist & line`, it reads how far the motion stretches it; the spring's stiffness is the dyad `line * (Twist & line) * k`, a map from twists to forques, and the plate's stiffness is the sum over its springs. The inertia is built the same way from the mass points, each joined with its own motion under a twist, `body.regressive(body.commutator(Twist))`, so no centre of mass or parallel-axis shift is needed. Paired back with a twist, both maps become energies, symmetric forms on twists, and the modes are the twists where the two energies are proportional: one eigenproblem of one form against the other.

In the matrix notation of structural dynamics the stiffness and inertia read as symmetric $3 \times 3$ matrices $K$ and $M$ on twist coordinates, and the modes as the generalized eigenproblem $K v = \lambda M v$, solved without forming $M^{-1} K$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from numga import NumpyContext, concatenate
from numga.algebras import PGA2D
from examples.util.animation import show_animation
from examples.mechanics.modes.render import (
    render_setup,
    draw_modes,
    animate_modes,
)
from examples.mechanics.modes.core import ModeCase

# The projective geometric algebra of space.
ctx = NumpyContext(PGA2D)
mv = ctx.multivector

Scalar = PGA2D.gatype.scalar()                    # Scalar+
Point = PGA2D.gatype.antivector()                 # Antivector-
# Rigid motions and velocities are twists; forces, torques and momenta are forques, their duals.
Twist = PGA2D.gatype.bivector()                   # Bivector+
Forque = PGA2D.gatype.antibivector()              # Antibivector-

## 1. The plate and its springs

A two-by-one plate of unit mass, its mass at the four corners, hangs from two vertical springs. Each spring is the line joining its anchor to its attachment.

In [ ]:
# The plate's corners, a two-by-one rectangle about (1, 1), and a quarter of its mass at each.
body = mv.point([[0.0, 0.5], [2.0, 0.5], [2.0, 1.5], [0.0, 1.5]])             # [corners] Point-
masses = np.full(4, 0.25)                                                      # [corners] kg

# Two vertical springs: their attachments on the plate, their anchors above, and their constants.
attachments = mv.point([[0.2, 1.5], [1.8, 1.5]])                                # [springs] Point-
anchors = mv.point([[0.2, 2.55], [1.8, 2.55]])                                  # [springs] Point-
spring_constants = np.array([6.0, 6.0])                                        # [springs] N/m

# Each spring's line, from its anchor to its attachment.
lines = (anchors & attachments).normalized()                                   # [springs] Forque-

fig = render_setup(
    body=body,
    anchors=anchors,
    attachments=attachments,
)
plt.show()

## 2. The inertia from mass points

Each mass point moves under a twist by its commutator with it, a map from twists to its velocity. Joined with that velocity and scaled by its mass, the point gives its line of momentum, a map from twists to forques; the sum over the points is the plate's inertia, its total mass, centre and moment together.

In [ ]:
# Each point's velocity as a map of the twist.
velocities = body.commutator(Twist)                                            # [corners] Point- <- Twist+

# Each point's momentum: the line through it along its velocity, its strength the mass times the speed.
point_momenta = (body & velocities) * masses                                   # [corners] Forque- <- Twist+

# The plate's inertia, the sum over its points.
inertia = point_momenta.sum(axis=0)                                            # [] Forque- <- Twist+

print(inertia)

## 3. The stiffness from springs

A spring paired with a twist reads how far the motion stretches it, a form on twists. The spring pulls back along its own line by that stretch times its constant, a dyad from twists to forques, and the plate's stiffness is the sum over its springs.

In [ ]:
# How far a twist stretches each spring: the twist paired with the spring's line.
extension = Twist & lines                                                      # [springs] Scalar+ <- Twist+

# Each spring pulls back along its line by its stretch times its constant, Hooke's law.
spring_stiffness = lines * extension * spring_constants                         # [springs] Forque- <- Twist+

# The plate's stiffness, the sum over its springs.
stiffness = spring_stiffness.sum(axis=0)                                       # [] Forque- <- Twist+

print(stiffness)

## 4. Energies and modes

Stiffness and inertia send a twist to a forque, a different kind of object, so neither has eigenvalues of its own. Paired back with a second twist, each becomes an energy, a symmetric form on two twists, and the natural modes are the twists where the two energies are proportional, `potential_form.eigh(kinetic_form)`. Held by two vertical springs, the plate slides sideways freely, at zero frequency, and bounces and rocks each on its own.

In [ ]:
# Each map paired with a second twist is an energy, a form on two twists, twice the energy on the same
# twist twice. Each spring gives (a & line) * (b & line), the same for a and b swapped, so the form is
# symmetric; so is the kinetic one.
potential_form = Twist & stiffness                                                    # [] Scalar+ <- (Twist+, Twist+)
kinetic_form = Twist & inertia                                                      # [] Scalar+ <- (Twist+, Twist+)

# The natural modes: the twists where the two energies are proportional.
values, modes = potential_form.eigh(kinetic_form)                                          # [modes] Scalar+, Twist+

# The frequencies in hertz: the square root of each eigenvalue, over a full turn.
frequencies = values.clip(0, np.inf).square_root() / (2 * np.pi)               # [modes] Scalar+

print("=== Baseline Suspension: Two Vertical Springs ===")
for label, freq in zip(["Free slide", "Rock", "Bounce"], frequencies.to_array()):
    detail = "0.00 Hz (free mechanism)" if freq == 0 else f"{freq:.3f} Hz"
    print(f"  {label:<16}: {detail}")

# How each mode moves the corners and the attachments, and stretches the springs.
body_offsets = body[None, :].commutator(modes[:, None])                        # [modes, corners] Point-
attachment_offsets = attachments[None, :].commutator(modes[:, None])          # [modes, springs] Point-
extensions = extension(modes[:, None])                                         # [modes, springs] Scalar+

case_baseline = ModeCase(body, attachments, anchors, frequencies, body_offsets, attachment_offsets, extensions)
fig = draw_modes([case_baseline])
plt.show()

## 5. An angled spring added

A third spring, set at an angle on the plate's side, holds it sideways too. Its stiffness adds to the others: the free slide is gone, and every mode mixes sliding, bouncing and rocking.

In [ ]:
# A third spring, at an angle on the plate's side, as stiff as the others.
attachment_extra = mv.point([2.0, 1.0])                                       # [] Point-
anchor_extra = mv.point([2.9, 1.85])                                          # [] Point-
constant_extra = 6.0
line_extra = (anchor_extra & attachment_extra).normalized()                   # [] Forque-
extension_extra = Twist & line_extra                                           # [] Scalar+ <- Twist+
spring_extra = line_extra * constant_extra * extension_extra                  # [] Forque- <- Twist+

# Its stiffness adds to the others.
stiffness_coupled = stiffness + spring_extra                                   # [] Forque- <- Twist+

# The new potential energy, solved against the same kinetic energy.
potential_form_coupled = Twist & stiffness_coupled                                    # [] Scalar+ <- (Twist+, Twist+)
values_coupled, modes_coupled = potential_form_coupled.eigh(kinetic_form)                  # [modes] Scalar+, Twist+
frequencies_coupled = values_coupled.clip(0, np.inf).square_root() / (2 * np.pi)  # [modes] Scalar+

print("=== Coupled Suspension: Added Angled Spring ===")
for label, freq in zip(["Coupled mode 1", "Coupled mode 2", "Coupled mode 3"], frequencies_coupled.to_array()):
    print(f"  {label:<16}: {freq:.3f} Hz")

# All three springs, for drawing.
attachments_coupled = concatenate([attachments, attachment_extra.reshape(1)])   # [springs] Point-
anchors_coupled = concatenate([anchors, anchor_extra.reshape(1)])             # [springs] Point-
lines_coupled = (anchors_coupled & attachments_coupled).normalized()           # [springs] Forque-
extension_coupled = Twist & lines_coupled                                      # [springs] Scalar+ <- Twist+

# How each coupled mode moves the corners and the attachments, and stretches the springs.
body_offsets_coupled = body[None, :].commutator(modes_coupled[:, None])        # [modes, corners] Point-
attachment_offsets_coupled = attachments_coupled[None, :].commutator(modes_coupled[:, None])  # [modes, springs] Point-
extensions_coupled = extension_coupled(modes_coupled[:, None])                 # [modes, springs] Scalar+

case_coupled = ModeCase(body, attachments_coupled, anchors_coupled, frequencies_coupled, body_offsets_coupled,
                        attachment_offsets_coupled, extensions_coupled)

fig = draw_modes([case_coupled])
plt.show()

## 6. The modes in motion

Released from rest in each coupled mode, the plate swings in that pattern alone.

In [ ]:
# The frames of the animation, and the duration of a frame.
n_frames, duration_ms = 150, 40
show_animation(animate_modes([case_coupled], n_frames), "modes", duration_ms)

## Summary

Every spring and every mass point contributes one map from twists to forques, and the plate's stiffness and inertia are their sums: adding a spring is adding its dyad, and the inertia carries the total mass, the centre and the moment with no reference point chosen. Paired back with a twist, the two maps are the potential and kinetic energies, and the modes are one eigenproblem of one energy against the other, `potential_form.eigh(kinetic_form)`, with no inverse of the inertia formed. A mode's motion of any point on the plate is the point's commutator with the mode's twist.

In [ ]:
# checks
a, b = modes[0] + modes[1], modes[2]
# Reciprocity: both energy forms are symmetric.
np.testing.assert_allclose(potential_form(a, b).to_array(), potential_form(b, a).to_array())
np.testing.assert_allclose(kinetic_form(a, b).to_array(), kinetic_form(b, a).to_array())